In [ ]:
!pip install ultralytics roboflow -q

In [ ]:
# Make sure to install roboflow if you haven't already
!pip install roboflow -q

from roboflow import Roboflow

rf = Roboflow(api_key="*********")
project = rf.workspace("visually-impaired-obstacle-detection-uxdze")\
            .project("obstacle-detection-yeuzf")
dataset = project.version(7).download("yolov8")

In [ ]:
import os

for item in os.listdir("/content/Obstacle-detection-7"):
    print(item)

In [ ]:
import cv2
import os
import shutil
import numpy as np
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

SRC  = "/content/Obstacle-detection-7"
DST  = "/content/Obstacle-processed-v2"

splits = ["train", "valid", "test"]

for split in splits:
    Path(f"{DST}/{split}/images").mkdir(parents=True, exist_ok=True)
    Path(f"{DST}/{split}/labels").mkdir(parents=True, exist_ok=True)

def preprocess(img):
    lab   = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    img = cv2.GaussianBlur(img, (3, 3), 0)
    return img

total = 0
for split in splits:
    img_src = Path(f"{SRC}/{split}/images")
    lbl_src = Path(f"{SRC}/{split}/labels")
    img_dst = Path(f"{DST}/{split}/images")
    lbl_dst = Path(f"{DST}/{split}/labels")

    if not img_src.exists():
        print(f"⚠️  {split}/images bulunamadı, atlandı.")
        continue

    imgs = list(img_src.glob("*.jpg")) + list(img_src.glob("*.png"))
    for img_path in imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        processed = preprocess(img)
        cv2.imwrite(str(img_dst / img_path.name), processed)
        lbl_file = lbl_src / (img_path.stem + ".txt")
        if lbl_file.exists():
            shutil.copy(lbl_file, lbl_dst / lbl_file.name)

    total += len(imgs)
    print(f"✅ {split}: {len(imgs)} görüntü işlendi")

yaml_src = Path(f"{SRC}/data.yaml")
yaml_dst = Path(f"{DST}/data.yaml")
with open(yaml_src, "r") as f:
    yaml_content = f.read()
yaml_content = yaml_content.replace(SRC, DST)
with open(yaml_dst, "w") as f:
    f.write(yaml_content)

print(f"\n🎉 Ön işlem tamamlandı! Toplam {total} görüntü işlendi.")



In [ ]:
import shutil
shutil.copytree("/content/Obstacle-processed-v2",
                "/content/drive/MyDrive/gorme_engelli/Obstacle-processed-v2",
                dirs_exist_ok=True)
print("✅ Drive'a kaydedildi!")

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/gorme_engelli/best.pt')

results = model.train(
    data='/content/Obstacle-processed-v2/data.yaml',
    epochs=50,
    imgsz=416,
    batch=64,
    workers=8,
    device=0,
    name="obstacle_v2_processed",
    project="/content/drive/MyDrive/gorme_engelli",
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache=True,
)

print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# gorme_engelli klasörünün tüm içeriğini listele
gorme_engelli = "/content/drive/MyDrive/gorme_engelli"

for item in os.listdir(gorme_engelli):
    print(item)

In [ ]:
import os

# Ön işlenmiş dataset var mı?
processed = "/content/drive/MyDrive/gorme_engelli/Obstacle-processed-v2"
print("=== Obstacle-processed-v2 ===")
for item in os.listdir(processed):
    print(item)

# yolov8m eğitimi tamamlandı mı?
yolov8m = "/content/drive/MyDrive/gorme_engelli/obstacle_v2_yolov8m"
print("\n=== obstacle_v2_yolov8m ===")
for item in os.listdir(yolov8m):
    print(item)

In [ ]:
import os
!pip install ultralytics -q

from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # nano yerine medium model

results = model.train(
    data='/content/Obstacle-processed-v2/data.yaml',
    epochs=50,
    imgsz=416,
    batch=64,
    workers=8,
    device=0,
    name="obstacle_v2_yolov8m",
    project="/content/drive/MyDrive/gorme_engelli",
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache=True,
)

print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/gorme_engelli/obstacle_v2_yolov8m-4/weights/best.pt')

results = model.train(
    data='/content/Obstacle-processed-v2/data.yaml',
    epochs=100,
    imgsz=416,
    batch=64,
    workers=8,
    device=0,
    name="obstacle_yolov8m_100ep",
    project="/content/drive/MyDrive/gorme_engelli",
    optimizer="AdamW",
    lr0=0.0005,
    patience=20,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache=True,
)

print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

In [ ]:
# Önce kütüphaneyi kuruyoruz (Sorunu çözen kısım burası)
!pip install roboflow

# Sonra indirme işlemine geçiyoruz
from roboflow import Roboflow
rf = Roboflow(api_key="*************")
project = rf.workspace("fpn").project("ood-pbnro")
dataset = project.version(1).download("yolov8")
print("OOD dataset indirildi ✅")

In [ ]:
import os

dataset_path = "/content/OOD-1"

for split in ["train", "valid", "test"]:
    path = f"{dataset_path}/{split}/images"
    if os.path.exists(path):
        print(f"{split}: {len(os.listdir(path))} görüntü")
    else:
        print(f"{split}: ❌ bulunamadı")

In [ ]:
import os
for item in os.listdir("/content"):
    print(item)

In [ ]:
import cv2
import os
import shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

SRC = "/content/OOD-1"
DST = "/content/OOD-processed"

splits = ["train", "valid", "test"]

for split in splits:
    Path(f"{DST}/{split}/images").mkdir(parents=True, exist_ok=True)
    Path(f"{DST}/{split}/labels").mkdir(parents=True, exist_ok=True)

def preprocess(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    img = cv2.GaussianBlur(img, (3, 3), 0)
    return img

total = 0
for split in splits:
    img_src = Path(f"{SRC}/{split}/images")
    lbl_src = Path(f"{SRC}/{split}/labels")
    img_dst = Path(f"{DST}/{split}/images")
    lbl_dst = Path(f"{DST}/{split}/labels")

    imgs = list(img_src.glob("*.jpg")) + list(img_src.glob("*.png"))
    for img_path in imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        processed = preprocess(img)
        cv2.imwrite(str(img_dst / img_path.name), processed)
        lbl_file = lbl_src / (img_path.stem + ".txt")
        if lbl_file.exists():
            shutil.copy(lbl_file, lbl_dst / lbl_file.name)

    total += len(imgs)
    print(f"✅ {split}: {len(imgs)} görüntü işlendi")

yaml_src = Path(f"{SRC}/data.yaml")
yaml_dst = Path(f"{DST}/data.yaml")
with open(yaml_src, "r") as f:
    yaml_content = f.read()
yaml_content = yaml_content.replace(SRC, DST)
with open(yaml_dst, "w") as f:
    f.write(yaml_content)

print(f"\n🎉 Ön işlem tamamlandı! Toplam {total} görüntü işlendi.")

# Drive'a kaydet
print("\n📦 Drive'a kopyalanıyor, bekleniyor...")
shutil.copytree(DST,
                "/content/drive/MyDrive/gorme_engelli/OOD-processed",
                dirs_exist_ok=True)
print("✅ Drive'a kaydedildi!")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # sıfırdan, OOD farklı sınıflar içeriyor

results = model.train(
    data='/content/OOD-processed/data.yaml',
    epochs=100,
    imgsz=416,
    batch=64,
    workers=8,
    device=0,
    name="ood_yolov8m",
    project="/content/drive/MyDrive/gorme_engelli",
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache=True,
)

print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

model_path = '/content/drive/MyDrive/gorme_engelli/obstacle_v2_yolov8m-4/weights/best.pt'
data_path  = '/content/Obstacle-processed-v2/data.yaml'

model = YOLO(model_path)

# ── 1. Validation metrikleri ──────────────────────────────────
results = model.val(data=data_path, imgsz=416, device=0, verbose=True)

print("\n" + "="*50)
print("📊 GENEL METRİKLER")
print("="*50)
print(f"mAP50      : {results.box.map50:.4f}")
print(f"mAP50-95   : {results.box.map:.4f}")
print(f"Precision  : {results.box.mp:.4f}")
print(f"Recall     : {results.box.mr:.4f}")

# ── 2. Sınıf bazlı metrikler ──────────────────────────────────
print("\n" + "="*50)
print("📋 SINIF BAZLI METRİKLER")
print("="*50)

names  = model.names
ap50   = results.box.ap50
ap     = results.box.ap

df = pd.DataFrame({
    'Sınıf'    : [names[i] for i in range(len(names))],
    'AP50'     : [round(float(ap50[i]), 4) for i in range(len(ap50))],
    'AP50-95'  : [round(float(ap[i]), 4)   for i in range(len(ap))],
})
df = df.sort_values('AP50', ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

# ── 3. Sınıf bazlı grafik ─────────────────────────────────────
plt.figure(figsize=(12, 6))
plt.barh(df['Sınıf'], df['AP50'], color='steelblue')
plt.axvline(x=results.box.map50, color='red', linestyle='--', label=f'Ortalama mAP50: {results.box.map50:.4f}')
plt.xlabel('AP50')
plt.title('Sınıf Bazlı AP50 Skorları (obstacle_v2_yolov8m)')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/gorme_engelli/sinif_bazli_ap50.png', dpi=150)
plt.show()
print("✅ Grafik Drive'a kaydedildi!")

# ── 4. Eğitim grafiği (results.csv'den) ──────────────────────
csv_path = '/content/drive/MyDrive/gorme_engelli/obstacle_v2_yolov8m-4/results.csv'
df_train = pd.read_csv(csv_path)
df_train.columns = df_train.columns.str.strip()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(df_train['epoch'], df_train['metrics/mAP50(B)'], label='mAP50')
plt.plot(df_train['epoch'], df_train['metrics/mAP50-95(B)'], label='mAP50-95')
plt.xlabel('Epoch')
plt.ylabel('mAP')
plt.title('Eğitim Boyunca mAP')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(df_train['epoch'], df_train['train/box_loss'], label='Train Loss')
plt.plot(df_train['epoch'], df_train['val/box_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Eğitim & Validation Loss')
plt.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/gorme_engelli/egitim_grafigi.png', dpi=150)
plt.show()
print("✅ Eğitim grafiği Drive'a kaydedildi!")

In [ ]:
!pip install ultralytics -q

import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

from ultralytics import YOLO
print("✅ Ultralytics hazır")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Dosya yolunu kontrol edin
model_path = '/content/drive/MyDrive/gorme_engelli/v1/weights/best.pt'

if not os.path.exists(model_path):
    print(f"HATA: Model dosyası bulunamadı: {model_path}")
    # Hata ayıklamaya yardımcı olmak için dizin içeriğini listeleyelim
    weights_dir = os.path.dirname(model_path)
    if os.path.exists(weights_dir):
        print(f"'{weights_dir}' dizinindeki dosyalar: {os.listdir(weights_dir)}")
    else:
        print(f"HATA: '{weights_dir}' dizini de bulunamadı.")
else:
    from ultralytics import YOLO
    model = YOLO(model_path)
    print("Model hazır ✅")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print("Model yüklendi ✅")

In [ ]:
results = model('https://ultralytics.com/images/bus.jpg', conf=0.4)
results[0].show()
print("Tespit edilen nesneler:", results[0].boxes.cls)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="TM9Hn2gSdzGPKTbn2Bsz")

project = rf.workspace("visually-impaired-obstacle-detection-uxdze") \
            .project("obstacle-detection-yeuzf")

# Mevcut versiyonları gör:
print(project.versions())

In [ ]:
dataset = project.version(7).download("yolov8")
print("Dataset indirildi ✅")

In [ ]:
import yaml

with open("/content/Obstacle-detection-7/data.yaml") as f:
    data = yaml.safe_load(f)

print("Sınıflar:", data['names'])
print("Sınıf sayısı:", data['nc'])

In [ ]:
import os
from collections import Counter

label_dir = "/content/Obstacle-detection-7/train/labels"
classes = []

for f in os.listdir(label_dir):
    with open(f"{label_dir}/{f}") as file:
        for line in file:
            classes.append(int(line.split()[0]))

sinif_isimleri = ['Bicycle','Bus','Car','Dog','Electric pole',
                  'Motorcycle','Person','Traffic signs','Tree','Uncovered manhole']

print("Sınıf dağılımı:")
for idx, count in sorted(Counter(classes).items()):
    print(f"  {sinif_isimleri[idx]}: {count} görüntü")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/Obstacle-detection-7/data.yaml',
    epochs=20,
    imgsz=416,
    batch=32,
    cache=True,
    amp=True,
    patience=10,
    project='gorme_engelli',
    name='v1'
)

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/drive/MyDrive/gorme_engelli', exist_ok=True)
shutil.copytree(
    '/content/runs/detect/gorme_engelli/v1',
    '/content/drive/MyDrive/gorme_engelli/v1',
    dirs_exist_ok=True
)
print("Drive'a kaydedildi ✅")

In [ ]:
results = model('https://ultralytics.com/images/bus.jpg', conf=0.4)
results[0].show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

results = model('https://ultralytics.com/images/bus.jpg', conf=0.4)
img = results[0].plot()
plt.figure(figsize=(10,8))
plt.imshow(img[:,:,::-1])
plt.axis('off')
plt.show()

In [ ]:
results = model('/content/drive/MyDrive/bisiklet-yol.jpg', conf=0.4)

from PIL import Image
import matplotlib.pyplot as plt
img = results[0].plot()
plt.figure(figsize=(10,8))
plt.imshow(img[:,:,::-1])
plt.axis('off')
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

results = model('/content/drive/MyDrive/bisiklet-yol.jpg', conf=0.4)
img = results[0].plot()
plt.figure(figsize=(10,8))
plt.imshow(img[:,:,::-1])
plt.axis('off')
plt.show()

In [ ]:
import os
if os.path.exists('/content/uyari2.mp3'):
    os.remove('/content/uyari2.mp3')

from gtts import gTTS
from IPython.display import Audio

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik direği',
    'motorcycle': 'motosiklet',
    'person': 'kişi',
    'traffic signs': 'trafik işareti',
    'tree': 'ağaç',
    'uncovered manhole': 'açık rögar'
}

results = model('/content/drive/MyDrive/bisiklet-yol.jpg', conf=0.4)
boxes = results[0].boxes
names = results[0].names

tespitler = []
for box in boxes:
    sinif = names[int(box.cls)].lower()  # küçük harfe çevir
    sinif_tr = sinif_isimleri_tr.get(sinif, sinif)

    goruntu_genisligi = results[0].orig_shape[1]
    merkez_x = (box.xyxy[0][0] + box.xyxy[0][2]) / 2

    if merkez_x < goruntu_genisligi / 3:
        konum = "solunuzda"
    elif merkez_x < 2 * goruntu_genisligi / 3:
        konum = "önünüzde"
    else:
        konum = "sağınızda"

    tespitler.append(f"{konum} {sinif_tr} var")

if tespitler:
    mesaj = ". ".join(tespitler)
    print("Mesaj:", mesaj)
    tts = gTTS(mesaj, lang='tr', tld='com.tr')
    tts.save('/content/uyari3.mp3')
    display(Audio('/content/uyari3.mp3', autoplay=True))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

results = model('/content/drive/MyDrive/araçlı-yol.webp', conf=0.4)
img = results[0].plot()
plt.figure(figsize=(10,8))
plt.imshow(img[:,:,::-1])
plt.axis('off')
plt.show()

In [ ]:
import os
if os.path.exists('/content/uyari3.mp3'):
    os.remove('/content/uyari3.mp3')

from gtts import gTTS
from IPython.display import Audio

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik direği',
    'motorcycle': 'motosiklet',
    'person': 'kişi',
    'traffic signs': 'trafik işareti',
    'tree': 'ağaç',
    'uncovered manhole': 'açık rögar',
    'truck': 'kamyon'  # eklendi
}

results = model('/content/drive/MyDrive/araçlı-yol.webp', conf=0.4)
boxes = results[0].boxes
names = results[0].names

# Konum bazlı grupla — tekrarları önle
konum_nesneler = {'solunuzda': [], 'önünüzde': [], 'sağınızda': []}

for box in boxes:
    sinif = names[int(box.cls)].lower()
    sinif_tr = sinif_isimleri_tr.get(sinif, sinif)

    goruntu_genisligi = results[0].orig_shape[1]
    merkez_x = (box.xyxy[0][0] + box.xyxy[0][2]) / 2

    if merkez_x < goruntu_genisligi / 3:
        konum = 'solunuzda'
    elif merkez_x < 2 * goruntu_genisligi / 3:
        konum = 'önünüzde'
    else:
        konum = 'sağınızda'

    # Aynı nesneyi aynı konuma ekleme
    if sinif_tr not in konum_nesneler[konum]:
        konum_nesneler[konum].append(sinif_tr)

# Mesajı oluştur
tespitler = []
for konum, nesneler in konum_nesneler.items():
    if nesneler:
        nesne_listesi = ' ve '.join(nesneler)
        tespitler.append(f"{konum} {nesne_listesi} var")

if tespitler:
    mesaj = ". ".join(tespitler)
    print("Mesaj:", mesaj)
    tts = gTTS(mesaj, lang='tr', tld='com.tr')
    tts.save('/content/uyari4.mp3')
    display(Audio('/content/uyari4.mp3', autoplay=True))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

results = model('/content/drive/MyDrive/araçlı-yol.webp', conf=0.5)  # 0.4'ten 0.5'e çıkardık
img = results[0].plot()
plt.figure(figsize=(12,8))
plt.imshow(img[:,:,::-1])
plt.axis('off')
plt.show()

# Neyi nerede gördüğünü de yaz
for box in results[0].boxes:
    sinif = results[0].names[int(box.cls)]
    conf = float(box.conf)
    print(f"{sinif} — güven: %{conf*100:.0f}")

In [ ]:
import time

son_uyari_zamani = {}
COOLDOWN = 3  # aynı nesne için 3 saniye bekleme

def uyari_ver(sinif_tr, konum):
    anahtar = f"{konum}_{sinif_tr}"
    simdi = time.time()

    if anahtar not in son_uyari_zamani or \
       simdi - son_uyari_zamani[anahtar] > COOLDOWN:
        son_uyari_zamani[anahtar] = simdi
        return True
    return False

In [ ]:
import yaml
with open('/content/drive/MyDrive/gorme_engelli/v1/args.yaml') as f:
    data = yaml.safe_load(f)
print(data)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="TM9Hn2gSdzGPKTbn2Bsz")
project = rf.workspace("visually-impaired-obstacle-detection-uxdze")\
            .project("obstacle-detection-yeuzf")
dataset = project.version(7).download("yolov8")
print("Dataset indirildi ✅")

In [ ]:
from ultralytics import YOLO
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

model = YOLO('yolov8n.pt')
model.train(
    data='/content/Obstacle-detection-7/data.yaml',
    epochs=20,
    imgsz=416,
    batch=32,
    cache=True,
    amp=True,
    patience=10,
    project='gorme_engelli',
    name='v1'
)

# Eğitim biter bitmez otomatik kaydet
os.makedirs('/content/drive/MyDrive/gorme_engelli', exist_ok=True)
shutil.copy(
    '/content/runs/detect/gorme_engelli/v1/weights/best.pt',
    '/content/drive/MyDrive/gorme_engelli/best.pt'
)
print("Model Drive'a kaydedildi ✅")

In [ ]:
import cv2
from gtts import gTTS
from IPython.display import display, Audio
import time

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik direği',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik işareti',
    'tree': 'ağaç',
    'uncovered manhole': 'açık rögar',
    'truck': 'kamyon',
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

def yolda_mi(kisi_box, engel_box):
    """
    Engel kişinin yürüyüş yolunda mı?
    Kişinin x aralığına engelin merkezi giriyor mu?
    """
    kisi_sol = kisi_box[0]
    kisi_sag = kisi_box[2]
    engel_merkez_x = (engel_box[0] + engel_box[2]) / 2
    return kisi_sol <= engel_merkez_x <= kisi_sag

video = cv2.VideoCapture('/content/drive/MyDrive/sokak video.mp4')
son_uyari_zamani = 0
COOLDOWN = 4
kare_sayisi = 0
uyari_sayisi = 0

while video.isOpened():
    ret, kare = video.read()
    if not ret:
        break

    kare_sayisi += 1
    if kare_sayisi % 30 != 0:
        continue

    results = model(kare, conf=0.4, verbose=False)
    boxes = results[0].boxes
    names = results[0].names
    genislik = kare.shape[1]

    kisiler = []
    engel_listesi = []

    for box in boxes:
        sinif = names[int(box.cls)].lower()
        coords = box.xyxy[0].tolist()
        if sinif == 'person':
            kisiler.append(coords)
        elif sinif in TUM_ENGELLER:
            engel_listesi.append((sinif, coords))

    if not kisiler or not engel_listesi:
        continue

    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        continue

    kisi = kisiler[0]
    kisi_merkez_x = (kisi[0] + kisi[2]) / 2
    mesaj = None

    for engel_sinif, engel_coords in engel_listesi:

        # Engel yolda değilse tamamen geç
        if not yolda_mi(kisi, engel_coords):
            continue

        # Yoldaysa uyar
        engel_tr = sinif_isimleri_tr.get(engel_sinif, engel_sinif)
        engel_merkez_x = (engel_coords[0] + engel_coords[2]) / 2
        fark = engel_merkez_x - kisi_merkez_x

        if abs(fark) < genislik * 0.08:
            mesaj = f"dikkat, önünüzde {engel_tr} var, durun"
        elif fark < 0:
            mesaj = f"dikkat, solunuzda {engel_tr} var, sağa yönelin"
        else:
            mesaj = f"dikkat, sağınızda {engel_tr} var, sola yönelin"
        break

    if mesaj:
        print(f"Kare {kare_sayisi}: {mesaj}")
        son_uyari_zamani = simdi
        uyari_sayisi += 1
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        tts.save(f'/content/uyari_{uyari_sayisi}.mp3')
        display(Audio(f'/content/uyari_{uyari_sayisi}.mp3', autoplay=True))
        time.sleep(COOLDOWN)

video.release()
print("Tamamlandı ✅")

In [ ]:
import cv2
from gtts import gTTS
from IPython.display import display, Audio, clear_output
import time
from PIL import Image
import matplotlib.pyplot as plt

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik direği',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik işareti',
    'tree': 'ağaç',
    'uncovered manhole': 'açık rögar',
    'truck': 'kamyon'
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

video = cv2.VideoCapture('/content/drive/MyDrive/yol4.mp4')
son_uyari_zamani = 0
COOLDOWN = 4
kare_sayisi = 0
uyari_sayisi = 0

while video.isOpened():
    ret, kare = video.read()
    if not ret:
        break

    kare_sayisi += 1

    # Her 15 karede analiz et
    if kare_sayisi % 15 != 0:
        continue

    results = model(kare, conf=0.3, verbose=False)
    boxes = results[0].boxes
    names = results[0].names
    genislik = kare.shape[1]

    kisiler = []
    engel_listesi = []

    for box in boxes:
        sinif = names[int(box.cls)].lower()
        coords = box.xyxy[0].tolist()
        if sinif == 'person':
            kisiler.append(coords)
        elif sinif in TUM_ENGELLER:
            engel_listesi.append((sinif, coords))

    # Görüntüyü göster
    annotated = results[0].plot()
    clear_output(wait=True)
    plt.figure(figsize=(10, 6))
    plt.imshow(annotated[:, :, ::-1])
    plt.axis('off')
    plt.show()

    # Kişi yoksa geç
    if not kisiler or not engel_listesi:
        continue

    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        continue

    # Kişinin yürüyüş yolunda engel var mı?
    kisi = kisiler[0]
    kisi_sol = kisi[0]
    kisi_sag = kisi[2]
    kisi_merkez_x = (kisi_sol + kisi_sag) / 2
    mesaj = None

    for engel_sinif, engel_coords in engel_listesi:
        engel_merkez_x = (engel_coords[0] + engel_coords[2]) / 2

        # Engel kişinin genişlik aralığında mı?
        if kisi_sol <= engel_merkez_x <= kisi_sag:
            engel_tr = sinif_isimleri_tr.get(engel_sinif, engel_sinif)
            fark = engel_merkez_x - kisi_merkez_x

            if abs(fark) < genislik * 0.08:
                mesaj = f"dikkat, önünüzde {engel_tr} var, durun"
            elif fark < 0:
                mesaj = f"dikkat, solunuzda {engel_tr} var, sağa yönelin"
            else:
                mesaj = f"dikkat, sağınızda {engel_tr} var, sola yönelin"
            break

    if mesaj:
        print(f"⚠️ Kare {kare_sayisi}: {mesaj}")
        son_uyari_zamani = simdi
        uyari_sayisi += 1
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        tts.save(f'/content/uyari_{uyari_sayisi}.mp3')
        display(Audio(f'/content/uyari_{uyari_sayisi}.mp3', autoplay=True))

video.release()
print("Tamamlandı ✅")

In [ ]:
import cv2
from gtts import gTTS
from IPython.display import display, Audio
import time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 80

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik direği',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik işareti',
    'tree': 'ağaç',
    'uncovered manhole': 'açık rögar',
    'truck': 'kamyon'
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

video = cv2.VideoCapture('/content/drive/MyDrive/yol4.mp4')
son_uyari_zamani = 0
COOLDOWN = 4
kare_sayisi = 0
uyari_sayisi = 0

fig, ax = plt.subplots(figsize=(10, 6))
plt.ion()

while video.isOpened():
    ret, kare = video.read()
    if not ret:
        break
    kare_sayisi += 1
    if kare_sayisi % 15 != 0:
        continue
    results = model(kare, conf=0.3, verbose=False)
    boxes = results[0].boxes
    names = results[0].names
    genislik = kare.shape[1]
    kisiler = []
    engel_listesi = []
    for box in boxes:
        sinif = names[int(box.cls)].lower()
        coords = box.xyxy[0].tolist()
        if sinif == 'person':
            kisiler.append(coords)
        elif sinif in TUM_ENGELLER:
            engel_listesi.append((sinif, coords))
    annotated = results[0].plot()
    ax.clear()
    ax.imshow(annotated[:, :, ::-1])
    ax.axis('off')
    fig.canvas.draw()
    plt.pause(0.01)
    if not kisiler or not engel_listesi:
        continue
    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        continue
    kisi = kisiler[0]
    kisi_sol = kisi[0]
    kisi_sag = kisi[2]
    kisi_merkez_x = (kisi_sol + kisi_sag) / 2
    mesaj = None
    for engel_sinif, engel_coords in engel_listesi:
        engel_merkez_x = (engel_coords[0] + engel_coords[2]) / 2
        if kisi_sol <= engel_merkez_x <= kisi_sag:
            engel_tr = sinif_isimleri_tr.get(engel_sinif, engel_sinif)
            fark = engel_merkez_x - kisi_merkez_x
            if abs(fark) < genislik * 0.08:
                mesaj = "dikkat, onunuzde " + engel_tr + " var, durun"
            elif fark < 0:
                mesaj = "dikkat, solunuzda " + engel_tr + " var, saga yonelin"
            else:
                mesaj = "dikkat, saginizda " + engel_tr + " var, sola yonelin"
            break
    if mesaj:
        print("Kare " + str(kare_sayisi) + ": " + mesaj)
        son_uyari_zamani = simdi
        uyari_sayisi += 1
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        dosya = '/content/uyari_' + str(uyari_sayisi) + '.mp3'
        tts.save(dosya)
        display(Audio(dosya, autoplay=True))

plt.ioff()
video.release()
print("Tamamlandi")

In [ ]:
import cv2
from gtts import gTTS
from IPython.display import display, Audio
import os

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik diregi',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik isareti',
    'tree': 'agac',
    'uncovered manhole': 'acik rogar',
    'truck': 'kamyon'
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

video = cv2.VideoCapture('/content/drive/MyDrive/yol4.mp4')
fps = video.get(cv2.CAP_PROP_FPS)
genislik_video = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik_video = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter('/content/sonuc.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps, (genislik_video, yukseklik_video))

kare_sayisi = 0
uyari_sayisi = 0
son_uyari_kare = 0
COOLDOWN_KARE = int(fps * 3)

print("Analiz başladı...")

while video.isOpened():
    ret, kare = video.read()
    if not ret:
        break

    kare_sayisi += 1
    results = model(kare, conf=0.3, verbose=False)
    annotated = results[0].plot()
    out.write(annotated)

    if kare_sayisi % 15 != 0:
        continue
    if kare_sayisi - son_uyari_kare < COOLDOWN_KARE:
        continue

    boxes = results[0].boxes
    names = results[0].names
    kisiler = []
    engel_listesi = []

    for box in boxes:
        sinif = names[int(box.cls)].lower()
        coords = box.xyxy[0].tolist()
        if sinif == 'person':
            kisiler.append(coords)
        elif sinif in TUM_ENGELLER:
            engel_listesi.append((sinif, coords))

    if not kisiler or not engel_listesi:
        continue

    kisi = kisiler[0]
    kisi_sol = kisi[0]
    kisi_sag = kisi[2]
    kisi_genislik = kisi_sag - kisi_sol
    kisi_merkez_x = (kisi_sol + kisi_sag) / 2

    # Toleransı 2x genişlet
    yol_sol = kisi_sol - kisi_genislik
    yol_sag = kisi_sag + kisi_genislik

    mesaj = None

    for engel_sinif, engel_coords in engel_listesi:
        engel_merkez_x = (engel_coords[0] + engel_coords[2]) / 2

        if yol_sol <= engel_merkez_x <= yol_sag:
            engel_tr = sinif_isimleri_tr.get(engel_sinif, engel_sinif)
            fark = engel_merkez_x - kisi_merkez_x
            saniye = kare_sayisi / fps

            if abs(fark) < kisi_genislik * 0.5:
                mesaj = "dikkat onunuzde " + engel_tr + " var durun"
            elif fark < 0:
                mesaj = "dikkat solunuzda " + engel_tr + " var saga yonelin"
            else:
                mesaj = "dikkat saginizda " + engel_tr + " var sola yonelin"
            break

    if mesaj:
        saniye = kare_sayisi / fps
        print(f"[{saniye:.1f}s] Kare {kare_sayisi}: {mesaj}")
        son_uyari_kare = kare_sayisi
        uyari_sayisi += 1
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        dosya = '/content/uyari_' + str(uyari_sayisi) + '.mp3'
        tts.save(dosya)
        display(Audio(dosya, autoplay=False))

video.release()
out.release()
print("Bitti! Toplam uyari: " + str(uyari_sayisi))

In [ ]:
import cv2
from gtts import gTTS
from IPython.display import display, Audio, HTML
from base64 import b64encode
import os

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik diregi',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik isareti',
    'tree': 'agac',
    'uncovered manhole': 'acik rogar',
    'truck': 'kamyon'
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

video = cv2.VideoCapture('/content/drive/MyDrive/yol1.mp4')
fps = video.get(cv2.CAP_PROP_FPS)
genislik_video = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik_video = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter('/content/sonuc1.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps, (genislik_video, yukseklik_video))

kare_sayisi = 0
uyari_sayisi = 0
son_uyari_kare = 0
COOLDOWN_KARE = int(fps * 3)

print("Analiz başladı...")

while video.isOpened():
    ret, kare = video.read()
    if not ret:
        break

    kare_sayisi += 1
    results = model(kare, conf=0.3, verbose=False)
    annotated = results[0].plot()
    out.write(annotated)

    if kare_sayisi % 15 != 0:
        continue
    if kare_sayisi - son_uyari_kare < COOLDOWN_KARE:
        continue

    boxes = results[0].boxes
    names = results[0].names
    kisiler = []
    engel_listesi = []

    for box in boxes:
        sinif = names[int(box.cls)].lower()
        coords = box.xyxy[0].tolist()
        if sinif == 'person':
            kisiler.append(coords)
        elif sinif in TUM_ENGELLER:
            engel_listesi.append((sinif, coords))

    if not kisiler or not engel_listesi:
        continue

    mesaj = None

    for kisi in kisiler:
        kisi_sol = kisi[0]
        kisi_sag = kisi[2]
        kisi_genislik = kisi_sag - kisi_sol
        kisi_merkez_x = (kisi_sol + kisi_sag) / 2

        # Yürüyüş yolu = kişinin genişliğinin 1.5 katı
        yol_sol = kisi_sol - kisi_genislik * 0.5
        yol_sag = kisi_sag + kisi_genislik * 0.5

        for engel_sinif, engel_coords in engel_listesi:
            e_sol = engel_coords[0]
            e_sag = engel_coords[2]
            e_merkez_x = (e_sol + e_sag) / 2

            # Engelin merkezi veya herhangi bir kenarı yürüyüş yolunda mı?
            yolda = (yol_sol <= e_merkez_x <= yol_sag or
                     yol_sol <= e_sol <= yol_sag or
                     yol_sol <= e_sag <= yol_sag)

            if not yolda:
                continue

            engel_tr = sinif_isimleri_tr.get(engel_sinif, engel_sinif)
            fark = e_merkez_x - kisi_merkez_x

            if abs(fark) < kisi_genislik * 0.3:
                mesaj = "dikkat onunuzde " + engel_tr + " var durun"
            elif fark < 0:
                mesaj = "dikkat solunuzda " + engel_tr + " var saga yonelin"
            else:
                mesaj = "dikkat saginizda " + engel_tr + " var sola yonelin"
            break

        if mesaj:
            break

    if mesaj:
        saniye = kare_sayisi / fps
        print(f"[{saniye:.1f}s] {mesaj}")
        son_uyari_kare = kare_sayisi
        uyari_sayisi += 1
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        dosya = '/content/uyari_' + str(uyari_sayisi) + '.mp3'
        tts.save(dosya)
        display(Audio(dosya, autoplay=True))

video.release()
out.release()
print("Bitti! Toplam uyari: " + str(uyari_sayisi))

# Videoyu göster
print("Video yükleniyor...")
mp4 = open('/content/sonuc1.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML("""
<video width=640 controls>
  <source src="%s" type="video/mp4">
</video>
""" % data_url))

In [ ]:
# Codec'i değiştir
!ffmpeg -i /content/sonuc1.mp4 -vcodec libx264 /content/sonuc1_h264.mp4 -y -loglevel quiet
print("Dönüştürüldü ✅")

from IPython.display import display, HTML
from base64 import b64encode

mp4 = open('/content/sonuc1_h264.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML("""
<video width=640 controls>
  <source src="%s" type="video/mp4">
</video>
""" % data_url))

In [ ]:
import cv2
from gtts import gTTS
from IPython.display import display, Audio, HTML
from base64 import b64encode

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik diregi',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik isareti',
    'tree': 'agac',
    'uncovered manhole': 'acik rogar',
    'truck': 'kamyon'
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

video = cv2.VideoCapture('/content/drive/MyDrive/yol1.mp4')
fps = video.get(cv2.CAP_PROP_FPS)
genislik_video = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik_video = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter('/content/sonuc2.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps, (genislik_video, yukseklik_video))

# Yürüyüş yolu = görüntünün orta %40'ı
yol_sol = genislik_video * 0.3
yol_sag = genislik_video * 0.7

kare_sayisi = 0
uyari_sayisi = 0
son_uyari_kare = 0
COOLDOWN_KARE = int(fps * 3)

print("Analiz başladı...")

while video.isOpened():
    ret, kare = video.read()
    if not ret:
        break

    kare_sayisi += 1
    results = model(kare, conf=0.3, verbose=False)
    annotated = results[0].plot()
    out.write(annotated)

    if kare_sayisi % 15 != 0:
        continue
    if kare_sayisi - son_uyari_kare < COOLDOWN_KARE:
        continue

    boxes = results[0].boxes
    names = results[0].names
    mesaj = None

    for box in boxes:
        sinif = names[int(box.cls)].lower()
        if sinif not in TUM_ENGELLER:
            continue

        coords = box.xyxy[0].tolist()
        e_merkez_x = (coords[0] + coords[2]) / 2

        # Engel yürüyüş yolunda mı?
        if not (yol_sol <= e_merkez_x <= yol_sag):
            continue

        engel_tr = sinif_isimleri_tr.get(sinif, sinif)

        # Görüntünün tam ortasında mı, sağında mı solunda mı?
        orta = genislik_video / 2
        fark = e_merkez_x - orta

        if abs(fark) < genislik_video * 0.1:
            mesaj = "dikkat onunuzde " + engel_tr + " var durun"
        elif fark < 0:
            mesaj = "dikkat solunuzda " + engel_tr + " var saga yonelin"
        else:
            mesaj = "dikkat saginizda " + engel_tr + " var sola yonelin"
        break

    if mesaj:
        saniye = kare_sayisi / fps
        print(f"[{saniye:.1f}s] {mesaj}")
        son_uyari_kare = kare_sayisi
        uyari_sayisi += 1
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        dosya = '/content/uyari_' + str(uyari_sayisi) + '.mp3'
        tts.save(dosya)
        display(Audio(dosya, autoplay=True))

video.release()
out.release()
print("Bitti! Toplam uyari: " + str(uyari_sayisi))

!ffmpeg -i /content/sonuc2.mp4 -vcodec libx264 /content/sonuc2_h264.mp4 -y -loglevel quiet

mp4 = open('/content/sonuc2_h264.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML("""
<video width=640 controls>
  <source src="%s" type="video/mp4">
</video>
""" % data_url))

In [ ]:
import cv2
from gtts import gTTS
from IPython.display import display, Audio, HTML
from base64 import b64encode

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik diregi',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik isareti',
    'tree': 'agac',
    'uncovered manhole': 'acik rogar',
    'truck': 'kamyon'
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

video = cv2.VideoCapture('/content/drive/MyDrive/yol1.mp4')
fps = video.get(cv2.CAP_PROP_FPS)
genislik_video = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik_video = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter('/content/sonuc3.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps, (genislik_video, yukseklik_video))

yol_sol = genislik_video * 0.3
yol_sag = genislik_video * 0.7

kare_sayisi = 0
uyari_sayisi = 0
son_uyari_kare = 0
COOLDOWN_KARE = int(fps * 3)
aktif_mesaj = ""
aktif_mesaj_kare = 0

print("Analiz başladı...")

while video.isOpened():
    ret, kare = video.read()
    if not ret:
        break

    kare_sayisi += 1
    results = model(kare, conf=0.3, verbose=False)
    annotated = results[0].plot()

    # Aktif uyarı varsa ekrana yaz (3 saniye boyunca)
    if kare_sayisi - aktif_mesaj_kare < int(fps * 3) and aktif_mesaj:
        cv2.rectangle(annotated, (0, yukseklik_video-80),
                     (genislik_video, yukseklik_video), (0,0,0), -1)
        cv2.putText(annotated, aktif_mesaj, (10, yukseklik_video-25),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    out.write(annotated)

    if kare_sayisi % 15 != 0:
        continue
    if kare_sayisi - son_uyari_kare < COOLDOWN_KARE:
        continue

    boxes = results[0].boxes
    names = results[0].names
    mesaj = None

    for box in boxes:
        sinif = names[int(box.cls)].lower()
        if sinif not in TUM_ENGELLER:
            continue

        coords = box.xyxy[0].tolist()
        e_merkez_x = (coords[0] + coords[2]) / 2

        if not (yol_sol <= e_merkez_x <= yol_sag):
            continue

        engel_tr = sinif_isimleri_tr.get(sinif, sinif)
        orta = genislik_video / 2
        fark = e_merkez_x - orta

        if abs(fark) < genislik_video * 0.1:
            mesaj = "onunuzde " + engel_tr + " var durun"
        elif fark < 0:
            mesaj = "solda " + engel_tr + " saga yonelin"
        else:
            mesaj = "sagda " + engel_tr + " sola yonelin"
        break

    if mesaj:
        saniye = kare_sayisi / fps
        print(f"[{saniye:.1f}s] {mesaj}")
        son_uyari_kare = kare_sayisi
        aktif_mesaj = mesaj
        aktif_mesaj_kare = kare_sayisi
        uyari_sayisi += 1
        tts = gTTS("dikkat " + mesaj, lang='tr', tld='com.tr')
        dosya = '/content/uyari_' + str(uyari_sayisi) + '.mp3'
        tts.save(dosya)
        display(Audio(dosya, autoplay=True))

video.release()
out.release()
print("Bitti! Toplam uyari: " + str(uyari_sayisi))

!ffmpeg -i /content/sonuc3.mp4 -vcodec libx264 /content/sonuc3_h264.mp4 -y -loglevel quiet
print("Video hazır ✅")

mp4 = open('/content/sonuc3_h264.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML("""
<video width=640 controls>
  <source src="%s" type="video/mp4">
</video>
""" % data_url))

In [ ]:
!pip install flask flask-ngrok pyngrok gtts -q
from pyngrok import ngrok

# Ngrok token al: dashboard.ngrok.com → ücretsiz hesap aç → authtoken kopyala
!ngrok authtoken "3ERuXedDAwfDl8wfRp29uxxedNx_4iBJZceGuKS22Bhon3AZz"
print("Ngrok hazır ✅")

In [ ]:
from flask import Flask, Response, render_template_string
from pyngrok import ngrok
import cv2
import base64
from gtts import gTTS
import os
import time

app = Flask(__name__)

sinif_isimleri_tr = {
    'bicycle': 'bisiklet',
    'bus': 'otobüs',
    'car': 'araba',
    'dog': 'köpek',
    'electric pole': 'elektrik diregi',
    'motorcycle': 'motorsiklet',
    'traffic signs': 'trafik isareti',
    'tree': 'agac',
    'uncovered manhole': 'acik rogar',
    'truck': 'kamyon'
}

TUM_ENGELLER = {'bicycle', 'bus', 'car', 'dog', 'motorcycle',
                'truck', 'uncovered manhole', 'tree', 'electric pole'}

son_uyari_zamani = 0
COOLDOWN = 3

HTML_SAYFA = '''
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Görme Engelli Asistan</title>
    <style>
        body { margin: 0; background: #000; display: flex;
               flex-direction: column; align-items: center; }
        h2 { color: white; font-family: Arial; }
        #video { width: 100%; max-width: 640px; }
        #uyari { color: #ff0; font-size: 24px; font-family: Arial;
                 padding: 10px; text-align: center; min-height: 40px; }
        canvas { display: none; }
    </style>
</head>
<body>
    <h2>Engel Tespit Sistemi</h2>
    <video id="video" autoplay playsinline></video>
    <div id="uyari">Sistem hazır...</div>
    <canvas id="canvas"></canvas>

    <script>
        const video = document.getElementById('video');
        const canvas = document.getElementById('canvas');
        const uyariDiv = document.getElementById('uyari');
        const ctx = canvas.getContext('2d');

        // Kamerayı aç
        navigator.mediaDevices.getUserMedia({video: {facingMode: 'environment'}})
            .then(stream => { video.srcObject = stream; });

        // Her 1 saniyede kare gönder
        setInterval(() => {
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            ctx.drawImage(video, 0, 0);
            const frame = canvas.toDataURL('image/jpeg', 0.7);

            fetch('/analiz', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({frame: frame})
            })
            .then(r => r.json())
            .then(data => {
                if (data.mesaj) {
                    uyariDiv.textContent = data.mesaj;
                    // Sesli uyarı
                    const ses = new Audio('/ses/' + data.ses_id);
                    ses.play();
                }
            });
        }, 1000);
    </script>
</body>
</html>
'''

@app.route('/')
def index():
    return render_template_string(HTML_SAYFA)

@app.route('/analiz', methods=['POST'])
def analiz():
    global son_uyari_zamani

    from flask import request, jsonify
    import numpy as np

    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        return jsonify({'mesaj': None})

    # Kareyi al
    data = request.json['frame']
    data = data.split(',')[1]
    img_bytes = base64.b64decode(data)
    img_array = np.frombuffer(img_bytes, dtype=np.uint8)
    kare = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if kare is None:
        return jsonify({'mesaj': None})

    genislik = kare.shape[1]
    yol_sol = genislik * 0.3
    yol_sag = genislik * 0.7

    results = model(kare, conf=0.3, verbose=False)
    boxes = results[0].boxes
    names = results[0].names
    mesaj = None

    for box in boxes:
        sinif = names[int(box.cls)].lower()
        if sinif not in TUM_ENGELLER:
            continue

        coords = box.xyxy[0].tolist()
        e_merkez_x = (coords[0] + coords[2]) / 2

        if not (yol_sol <= e_merkez_x <= yol_sag):
            continue

        engel_tr = sinif_isimleri_tr.get(sinif, sinif)
        orta = genislik / 2
        fark = e_merkez_x - orta

        if abs(fark) < genislik * 0.1:
            mesaj = "dikkat onunuzde " + engel_tr + " var durun"
        elif fark < 0:
            mesaj = "dikkat solunuzda " + engel_tr + " var saga yonelin"
        else:
            mesaj = "dikkat saginizda " + engel_tr + " var sola yonelin"
        break

    if mesaj:
        son_uyari_zamani = simdi
        ses_id = str(int(simdi)) + '.mp3'
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        tts.save('/content/' + ses_id)
        return jsonify({'mesaj': mesaj, 'ses_id': ses_id})

    return jsonify({'mesaj': None})

@app.route('/ses/<ses_id>')
def ses(ses_id):
    from flask import send_file
    return send_file('/content/' + ses_id, mimetype='audio/mpeg')

# Sunucuyu başlat
public_url = ngrok.connect(5000)
print("Telefonda şu adresi aç:", public_url)
print("HTTPS ile aç, HTTP değil!")

app.run(port=5000)

In [ ]:
!pip install flask pyngrok gtts ultralytics -q
from pyngrok import ngrok
from google.colab import drive
from ultralytics import YOLO

# Önce modeli yükle
drive.mount('/content/drive')
model = YOLO('/content/drive/MyDrive/gorme_engelli/best.pt')
print("Model yüklendi ✅")

# Sonra Flask'ı başlat
from flask import Flask, request, jsonify, render_template_string
import cv2, base64, numpy as np, time
from gtts import gTTS
import logging
logging.basicConfig(level=logging.ERROR)

app = Flask(__name__)

sinif_isimleri_tr = {
    'bicycle':'bisiklet','bus':'otobüs','car':'araba',
    'dog':'köpek','electric pole':'elektrik diregi',
    'motorcycle':'motorsiklet','traffic signs':'trafik isareti',
    'tree':'agac','uncovered manhole':'acik rogar','truck':'kamyon'
}
TUM_ENGELLER = {'bicycle','bus','car','dog','motorcycle',
                'truck','uncovered manhole','tree','electric pole'}

son_uyari_zamani = 0
COOLDOWN = 3

HTML = '''<!DOCTYPE html><html><head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Engel Tespit</title>
<style>body{margin:0;background:#000;display:flex;flex-direction:column;align-items:center}
h2{color:white;font-family:Arial}
#video{width:100%;max-width:640px}
#uyari{color:#ff0;font-size:22px;font-family:Arial;padding:10px;text-align:center;min-height:40px}
canvas{display:none}</style></head>
<body><h2>Engel Tespit Sistemi</h2>
<video id="video" autoplay playsinline></video>
<div id="uyari">Sistem hazir...</div>
<canvas id="canvas"></canvas>
<script>
const video=document.getElementById('video');
const canvas=document.getElementById('canvas');
const uyariDiv=document.getElementById('uyari');
const ctx=canvas.getContext('2d');
navigator.mediaDevices.getUserMedia({video:{facingMode:'environment'}})
  .then(s=>{video.srcObject=s});
let busy=false;
setInterval(()=>{
  if(busy||!video.videoWidth) return;
  busy=true;
  canvas.width=video.videoWidth;
  canvas.height=video.videoHeight;
  ctx.drawImage(video,0,0);
  fetch('/analiz',{method:'POST',
    headers:{'Content-Type':'application/json'},
    body:JSON.stringify({frame:canvas.toDataURL('image/jpeg',0.6)})})
  .then(r=>r.json()).then(d=>{
    if(d.mesaj){
      uyariDiv.textContent=d.mesaj;
      new Audio('/ses/'+d.ses_id).play();
    }
    busy=false;
  }).catch(()=>{busy=false});
},1500);
</script></body></html>'''

@app.route('/')
def index():
    return render_template_string(HTML)

@app.route('/analiz', methods=['POST'])
def analiz():
    global son_uyari_zamani
    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        return jsonify({'mesaj': None})
    data = request.json['frame'].split(',')[1]
    img_bytes = base64.b64decode(data)
    img_array = np.frombuffer(img_bytes, dtype=np.uint8)
    kare = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    if kare is None:
        return jsonify({'mesaj': None})
    genislik = kare.shape[1]
    yol_sol = genislik * 0.3
    yol_sag = genislik * 0.7
    results = model(kare, conf=0.3, verbose=False)
    mesaj = None
    for box in results[0].boxes:
        sinif = results[0].names[int(box.cls)].lower()
        if sinif not in TUM_ENGELLER:
            continue
        coords = box.xyxy[0].tolist()
        e_x = (coords[0]+coords[2])/2
        if not (yol_sol <= e_x <= yol_sag):
            continue
        engel_tr = sinif_isimleri_tr.get(sinif, sinif)
        fark = e_x - genislik/2
        if abs(fark) < genislik*0.1:
            mesaj = "dikkat onunuzde " + engel_tr + " var durun"
        elif fark < 0:
            mesaj = "dikkat solunuzda " + engel_tr + " var saga yonelin"
        else:
            mesaj = "dikkat saginizda " + engel_tr + " var sola yonelin"
        break
    if mesaj:
        son_uyari_zamani = simdi
        ses_id = str(int(simdi)) + '.mp3'
        gTTS(mesaj, lang='tr', tld='com.tr').save('/content/'+ses_id)
        return jsonify({'mesaj': mesaj, 'ses_id': ses_id})
    return jsonify({'mesaj': None})

@app.route('/ses/<ses_id>')
def ses(ses_id):
    from flask import send_file
    return send_file('/content/'+ses_id, mimetype='audio/mpeg')

!ngrok authtoken "3ERuXedDAwfDl8wfRp29uxxedNx_4iBJZceGuKS22Bhon3AZz"
url = ngrok.connect(5000)
print("Telefonda aç:", url)
app.run(port=5000)

In [ ]:
import yaml
import os

yaml_yolu = 'data.yaml' # Eğer dosya başka bir klasördeyse yolunu buraya yaz

if os.path.exists(yaml_yolu):
    with open(yaml_yolu, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
    print("--- VERİ SETİ YAPISI ---")
    print(f"Sınıf Sayısı (nc): {data.get('nc')}")
    print(f"Sınıf İsimleri (names): {data.get('names')}")
    print(f"Eğitim (Train) Yolu: {data.get('train')}")
    print(f"Doğrulama (Val) Yolu: {data.get('val')}")
else:
    print("data.yaml dosyası bulunamadı. Lütfen dosya yolunu kontrol et.")

In [ ]:
from ultralytics import YOLO

# Eğittiğin modelin ağırlık dosyasını (weights) yükle.
# Eğer yolunu değiştirmediysen YOLOv8 varsayılan olarak sonuçları bu klasöre kaydeder:
model = YOLO('runs/detect/train/weights/best.pt')

# Modelin genel bilgilerini (parametre sayısı, katman sayısı vb.) yazdırır
print("Model Bilgileri:")
model.info()

# Modelin tanıdığı etiketleri (sınıfları) görmek istersen:
print("\nModelin Tanıdığı Sınıflar:")
print(model.names)

In [ ]:
!pip install ultralytics

In [ ]:
import yaml

with open("/content/OOD-1/data.yaml") as f:
    data = yaml.safe_load(f)

print("Sınıflar:", data['names'])
print("Sınıf sayısı:", data['nc'])

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/OOD-1/data.yaml',
    epochs=20,
    imgsz=416,
    batch=32,
    cache=True,
    amp=True,
    patience=10,
    project='gorme_engelli',
    name='v2_ood'
)

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/gorme_engelli', exist_ok=True)
shutil.copy(
    '/content/runs/detect/gorme_engelli/v2_ood/weights/best.pt',
    '/content/drive/MyDrive/gorme_engelli/best_v2.pt'
)
print("Kaydedildi ✅")

In [ ]:
import pandas as pd

df = pd.read_csv('/content/runs/detect/gorme_engelli/v2_ood/results.csv')
df.columns = df.columns.str.strip()
print("Son epoch mAP50:", df['metrics/mAP50(B)'].iloc[-1])
print("En iyi mAP50:", df['metrics/mAP50(B)'].max())

In [ ]:
!pip install flask pyngrok gtts ultralytics -q
from pyngrok import ngrok
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')
model = YOLO('/content/drive/MyDrive/gorme_engelli/best_v2.pt')
print("Yeni model yüklendi ✅")

from flask import Flask, request, jsonify, render_template_string
import cv2, base64, numpy as np, time
from gtts import gTTS
import logging
logging.basicConfig(level=logging.ERROR)

app = Flask(__name__)

sinif_isimleri_tr = {
    'person':'kisi','car':'araba','tree':'agac',
    'stairs':'merdiven','bicycle':'bisiklet',
    'motorcycle':'motorsiklet','dog':'kopek',
    'bus':'otobus','truck':'kamyon',
    'fire_hydrant':'yangın musluğu','bench':'bank',
    'curb':'bordur','traffic_light':'trafik ışığı',
    'stop_sign':'dur levhası','pole':'direk',
    'bus_stop':'durak','spherical_roadblock':'yol bariyeri',
    'warning_column':'uyarı dubası',
    'waste_container':'cop kutusu',
    'street_light':'sokak lambası','crutch':'koltuk değneği'
}

TUM_ENGELLER = set(sinif_isimleri_tr.keys()) - {'person'}

son_uyari_zamani = 0
COOLDOWN = 3

HTML = '''<!DOCTYPE html><html><head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Engel Tespit</title>
<style>body{margin:0;background:#000;display:flex;flex-direction:column;align-items:center}
h2{color:white;font-family:Arial}
#video{width:100%;max-width:640px}
#uyari{color:#ff0;font-size:22px;font-family:Arial;padding:10px;text-align:center;min-height:40px}
canvas{display:none}</style></head>
<body><h2>Engel Tespit Sistemi</h2>
<video id="video" autoplay playsinline></video>
<div id="uyari">Sistem hazir...</div>
<canvas id="canvas"></canvas>
<script>
const video=document.getElementById('video');
const canvas=document.getElementById('canvas');
const uyariDiv=document.getElementById('uyari');
const ctx=canvas.getContext('2d');
navigator.mediaDevices.getUserMedia({video:{facingMode:'environment'}})
  .then(s=>{video.srcObject=s});
let busy=false;
setInterval(()=>{
  if(busy||!video.videoWidth) return;
  busy=true;
  canvas.width=video.videoWidth;
  canvas.height=video.videoHeight;
  ctx.drawImage(video,0,0);
  fetch('/analiz',{method:'POST',
    headers:{'Content-Type':'application/json'},
    body:JSON.stringify({frame:canvas.toDataURL('image/jpeg',0.6)})})
  .then(r=>r.json()).then(d=>{
    if(d.mesaj){uyariDiv.textContent=d.mesaj;}
    busy=false;
  }).catch(()=>{busy=false});
},1500);
</script></body></html>'''

@app.route('/')
def index():
    return render_template_string(HTML)

@app.route('/analiz', methods=['POST'])
def analiz():
    global son_uyari_zamani
    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        return jsonify({'mesaj': None})
    data = request.json['frame'].split(',')[1]
    kare = cv2.imdecode(np.frombuffer(base64.b64decode(data), np.uint8), cv2.IMREAD_COLOR)
    if kare is None:
        return jsonify({'mesaj': None})
    genislik = kare.shape[1]
    yol_sol = genislik * 0.3
    yol_sag = genislik * 0.7
    results = model(kare, conf=0.3, verbose=False)
    mesaj = None
    for box in results[0].boxes:
        sinif = results[0].names[int(box.cls)].lower()
        if sinif not in TUM_ENGELLER:
            continue
        coords = box.xyxy[0].tolist()
        e_x = (coords[0]+coords[2])/2
        if not (yol_sol <= e_x <= yol_sag):
            continue
        engel_tr = sinif_isimleri_tr.get(sinif, sinif)
        fark = e_x - genislik/2
        if abs(fark) < genislik*0.1:
            mesaj = "dikkat onunuzde " + engel_tr + " var durun"
        elif fark < 0:
            mesaj = "dikkat solunuzda " + engel_tr + " var saga yonelin"
        else:
            mesaj = "dikkat saginizda " + engel_tr + " var sola yonelin"
        break
    if mesaj:
        son_uyari_zamani = simdi
        tts = gTTS(mesaj, lang='tr', tld='com.tr')
        ses_id = str(int(simdi)) + '.mp3'
        tts.save('/content/'+ses_id)
        return jsonify({'mesaj': mesaj, 'ses_id': ses_id})
    return jsonify({'mesaj': None})

@app.route('/ses/<ses_id>')
def ses(ses_id):
    from flask import send_file
    return send_file('/content/'+ses_id, mimetype='audio/mpeg')

!ngrok authtoken "3ERuXedDAwfDl8wfRp29uxxedNx_4iBJZceGuKS22Bhon3AZz"
url = ngrok.connect(5000)
print("Telefonda aç:", url)
app.run(port=5000)

In [ ]:
!pip install flask pyngrok gtts ultralytics -q
from pyngrok import ngrok
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')
model = YOLO('/content/drive/MyDrive/gorme_engelli/best_v2.pt')
print("Model yüklendi ✅")

from flask import Flask, request, jsonify, render_template_string, send_file
import cv2, base64, numpy as np, time
from gtts import gTTS
import logging
logging.basicConfig(level=logging.ERROR)

app = Flask(__name__)

sinif_isimleri_tr = {
    'car':'araba','tree':'agac','stairs':'merdiven',
    'bicycle':'bisiklet','motorcycle':'motorsiklet',
    'dog':'kopek','bus':'otobus','truck':'kamyon',
    'fire_hydrant':'yangin muslugu','bench':'bank',
    'curb':'bordur','traffic_light':'trafik isigi',
    'stop_sign':'dur levhasi','pole':'direk',
    'bus_stop':'durak','spherical_roadblock':'yol bariyeri',
    'warning_column':'uyari dubasi',
    'waste_container':'cop kutusu',
    'street_light':'sokak lambasi','crutch':'koltuk degneği'
}

ZEMIN_ENGEL = {'stairs', 'curb'}
TUM_ENGELLER = set(sinif_isimleri_tr.keys())
son_uyari_zamani = 0
COOLDOWN = 3

HTML = '''<!DOCTYPE html><html><head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Engel Tespit</title>
<style>
  *{margin:0;padding:0;box-sizing:border-box}
  body{background:#000;width:100vw;height:100vh;overflow:hidden;display:flex;align-items:center;justify-content:center}
  #video{width:100vw;height:100vh;object-fit:cover;display:none}
  canvas{display:none}
  #baslat{
    background:#ff0;color:#000;border:none;
    font-size:28px;font-weight:bold;font-family:Arial;
    padding:30px 60px;border-radius:20px;cursor:pointer;
  }
</style></head>
<body>
<button id="baslat">BAŞLAT</button>
<video id="video" autoplay playsinline></video>
<canvas id="canvas"></canvas>
<script>
const video = document.getElementById('video');
const canvas = document.getElementById('canvas');
const ctx = canvas.getContext('2d');
const btn = document.getElementById('baslat');

// Sessiz ses — tarayıcının ses iznini unlock eder
const AudioContext = window.AudioContext || window.webkitAudioContext;
let audioCtx;

btn.addEventListener('click', () => {
  btn.style.display = 'none';
  video.style.display = 'block';

  // AudioContext başlat
  audioCtx = new AudioContext();

  // Kamerayı aç
  navigator.mediaDevices.getUserMedia({video:{facingMode:'environment'}})
    .then(s => { video.srcObject = s; });

  // Analiz döngüsü
  let busy = false;
  setInterval(() => {
    if(busy || !video.videoWidth) return;
    busy = true;
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    ctx.drawImage(video, 0, 0);

    fetch('/analiz', {
      method: 'POST',
      headers: {'Content-Type': 'application/json'},
      body: JSON.stringify({frame: canvas.toDataURL('image/jpeg', 0.6)})
    })
    .then(r => r.json())
    .then(d => {
      if(d.ses_id) {
        // AudioContext ile ses çal
        fetch('/ses/' + d.ses_id)
          .then(r => r.arrayBuffer())
          .then(buf => audioCtx.decodeAudioData(buf))
          .then(decoded => {
            const src = audioCtx.createBufferSource();
            src.buffer = decoded;
            src.connect(audioCtx.destination);
            src.start(0);
          }).catch(e => console.log('ses hatasi', e));
      }
      busy = false;
    }).catch(() => { busy = false; });
  }, 2000);
});
</script></body></html>'''

@app.route('/')
def index():
    return render_template_string(HTML)

@app.route('/analiz', methods=['POST'])
def analiz():
    global son_uyari_zamani
    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        return jsonify({'ses_id': None})

    data = request.json['frame'].split(',')[1]
    kare = cv2.imdecode(np.frombuffer(base64.b64decode(data), np.uint8), cv2.IMREAD_COLOR)
    if kare is None:
        return jsonify({'ses_id': None})

    genislik = kare.shape[1]
    yol_sol = genislik * 0.3
    yol_sag = genislik * 0.7

    results = model(kare, conf=0.5, verbose=False)
    mesaj = None

    for box in results[0].boxes:
        sinif = results[0].names[int(box.cls)].lower()
        if sinif not in TUM_ENGELLER:
            continue
        coords = box.xyxy[0].tolist()
        e_x = (coords[0] + coords[2]) / 2
        if not (yol_sol <= e_x <= yol_sag):
            continue

        engel_tr = sinif_isimleri_tr.get(sinif, sinif)

        if sinif in ZEMIN_ENGEL:
            mesaj = "dikkat onunuzde " + engel_tr + " var dikkatli olun"
        else:
            fark = e_x - genislik / 2
            if abs(fark) < genislik * 0.1:
                mesaj = "dikkat onunuzde " + engel_tr + " var durun"
            elif fark < 0:
                mesaj = "dikkat solunuzda " + engel_tr + " var saga yonelin"
            else:
                mesaj = "dikkat saginizda " + engel_tr + " var sola yonelin"
        break

    if mesaj:
        son_uyari_zamani = simdi
        ses_id = str(int(simdi)) + '.mp3'
        gTTS(mesaj, lang='tr', tld='com.tr').save('/content/' + ses_id)
        print("Uyari:", mesaj)
        return jsonify({'ses_id': ses_id})

    return jsonify({'ses_id': None})

@app.route('/ses/<ses_id>')
def ses(ses_id):
    return send_file('/content/' + ses_id, mimetype='audio/mpeg')

!ngrok authtoken "3ERuXedDAwfDl8wfRp29uxxedNx_4iBJZceGuKS22Bhon3AZz"
url = ngrok.connect(5000)
print("Telefonda aç:", url)
app.run(port=5000)

In [ ]:
import os
print([os.path.join(r, f) for r, d, fs in os.walk('/content') for f in fs if f == 'best.pt'])

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/gorme_engelli/best.pt')
# (Yukarıda çıkan yolu parantez içine yapıştırman yeterli)

In [ ]:
import os

# 1. Klasör isimlerine bakarak veri seti adını tahmin etme
print("--- 1. Colab'daki Klasörler ---")
folders = [f for f in os.listdir('/content') if os.path.isdir(os.path.join('/content', f)) and f != '.config' and f != 'runs']
print("Veri Seti Olma İhtimali Yüksek Klasörler:", folders)

# 2. data.yaml dosyasını bulup içeriğini okuma
print("\n--- 2. Veri Seti Detayları (data.yaml) ---")
yaml_path = None
for r, d, fs in os.walk('/content'):
    for f in fs:
        if f == 'data.yaml':
            yaml_path = os.path.join(r, f)
            break

if yaml_path:
    print(f"Bulunan data.yaml Yolu: {yaml_path}\n")
    with open(yaml_path, 'r') as file:
        print(file.read())
else:
    print("data.yaml dosyası bulunamadı.")

In [ ]:
from flask import Flask, request, jsonify, render_template_string, send_file
import cv2, base64, numpy as np, time
!pip install gtts -q
from gtts import gTTS
import logging
logging.basicConfig(level=logging.ERROR)

app = Flask(__name__)

# ... mevcut kodların aynı kalıyor ...

# ✅ Mobil uygulama için /predict endpoint'i ekle
@app.route('/predict', methods=['POST'])
def predict():
    global son_uyari_zamani
    simdi = time.time()
    if simdi - son_uyari_zamani < COOLDOWN:
        return jsonify({'detections': []})

    file = request.files.get('image')
    if not file:
        return jsonify({'detections': []})

    kare = cv2.imdecode(np.frombuffer(file.read(), np.uint8), cv2.IMREAD_COLOR)
    if kare is None:
        return jsonify({'detections': []})

    genislik = kare.shape[1]
    yol_sol = genislik * 0.3
    yol_sag = genislik * 0.7

    results = model(kare, conf=0.5, verbose=False)
    mesajlar = []

    for box in results[0].boxes:
        sinif = results[0].names[int(box.cls)].lower()
        if sinif not in TUM_ENGELLER:
            continue
        coords = box.xyxy[0].tolist()
        e_x = (coords[0] + coords[2]) / 2
        if not (yol_sol <= e_x <= yol_sag):
            continue

        engel_tr = sinif_isimleri_tr.get(sinif, sinif)

        if sinif in ZEMIN_ENGEL:
            mesaj = "dikkat önünüzde " + engel_tr + " var dikkatli olun"
        else:
            fark = e_x - genislik / 2
            if abs(fark) < genislik * 0.1:
                mesaj = "dikkat önünüzde " + engel_tr + " var durun"
            elif fark < 0:
                mesaj = "dikkat solunuzda " + engel_tr + " var sağa yönelin"
            else:
                mesaj = "dikkat sağınızda " + engel_tr + " var sola yönelin"

        mesajlar.append({"label": mesaj, "confidence": float(box.conf[0])})
        break  # En önemli engeli al

    if mesajlar:
        son_uyari_zamani = simdi

    return jsonify({'detections': mesajlar})

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/gorme_engelli/best_v2.pt')
print(model.names)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q
from ultralytics import YOLO
import os

# Drive'daki modelleri listele
klasor = '/content/drive/MyDrive/gorme_engelli'
for dosya in os.listdir(klasor):
    print(dosya)

In [ ]:
import pandas as pd
import os

file_path = '/content/runs/detect/gorme_engelli/v2_ood/results.csv'
directory_path = os.path.dirname(file_path)

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    print("Son epoch mAP50:", df['metrics/mAP50(B)'].iloc[-1])
    print("En iyi mAP50:", df['metrics/mAP50(B)'].max())
    print("Son epoch mAP50-95:", df['metrics/mAP50-95(B)'].iloc[-1])
    print("En iyi mAP50-95:", df['metrics/mAP50-95(B)'].max())
else:
    print(f"Hata: '{file_path}' dosyası bulunamadı.")
    # Check the parent directory for other run folders
    parent_directory = os.path.dirname(directory_path) # /content/runs/detect/gorme_engelli/
    if os.path.exists(parent_directory):
        print(f"'{parent_directory}' dizinindeki içerik: {os.listdir(parent_directory)}")
    else:
        print(f"Hata: '{parent_directory}' dizini de bulunamadı.")

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/gorme_engelli/best_v2.pt')
print(model.info())

In [ ]:
import os
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import drive

# Drive'ı bağla - bitince hemen kaydedelim
drive.mount('/content/drive')

rf = Roboflow(api_key="TM9Hn2gSdzGPKTbn2Bsz")
project = rf.workspace("fpn").project("ood-pbnro")
dataset = project.version(1).download("yolov8")
print("Dataset indirildi ✅")

model = YOLO('yolov8s.pt')  # nano yerine small — daha iyi merdiven tespiti

# Correcting the data path
TRAIN_DATA_PATH = os.path.join(dataset.location, 'data.yaml')

model.train(
    data=TRAIN_DATA_PATH,
    epochs=30,        # 100 değil, kota gitmez
    imgsz=512,        # 640 değil, 416'dan iyi ama kota dostu
    batch=16,
    cache=True,
    amp=True,
    patience=10,
    project='gorme_engelli',
    name='v_ood_s'
)

# Hemen kaydet
os.makedirs('/content/drive/MyDrive/gorme_engelli', exist_ok=True)
import shutil
shutil.copy(
    '/content/runs/detect/gorme_engelli/v_ood_s/weights/best.pt',
    '/content/drive/MyDrive/gorme_engelli/best_ood_s.pt'
)
print("Drive'a kaydedildi ✅")

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="TM9Hn2gSdzGPKTbn2Bsz")

# Roboflow'da stairs detection dataseti
project = rf.workspace("fpn").project("ood-pbnro") # Corrected to a working project
dataset = project.version(1).download("yolov8")
print("Merdiven dataseti indirildi ✅")

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/gorme_engelli/best.pt') # Corrected path to the trained model
print("Sınıflar:", model.names)

In [ ]:
!pip install roboflow -q
from roboflow import Roboflow
from google.colab import drive
from ultralytics import YOLO
import shutil, os

drive.mount('/content/drive')

rf = Roboflow(api_key="TM9Hn2gSdzGPKTbn2Bsz")
project = rf.workspace("barrierfreenavigation").project("stairs-u437q")
version = project.version(2)
dataset = version.download("yolov8")
print("Dataset indirildi ✅")
print("Konum:", dataset.location)

In [ ]:
model = YOLO('/content/drive/MyDrive/gorme_engelli/best.pt')

model.train(
    data=dataset.location + '/data.yaml',
    epochs=20,
    imgsz=416,
    batch=32,
    cache=True,
    amp=True,
    patience=10,
    project='gorme_engelli',
    name='v_merdiven'
)

# Bitince hemen kaydet
os.makedirs('/content/drive/MyDrive/gorme_engelli', exist_ok=True)
shutil.copy(
    '/content/runs/detect/gorme_engelli/v_merdiven/weights/best.pt',
    '/content/drive/MyDrive/gorme_engelli/best_merdiven.pt'
)
print("Kaydedildi ✅")

In [ ]:
import pandas as pd

df = pd.read_csv('/content/runs/detect/gorme_engelli/v_merdiven/results.csv')
df.columns = df.columns.str.strip()
print("Son mAP50:", df['metrics/mAP50(B)'].iloc[-1])
print("En iyi mAP50:", df['metrics/mAP50(B)'].max())

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/gorme_engelli/best_merdiven.pt')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/gorme_engelli/best.pt')

# Sınıfları gör
print("Sınıflar:", model.names)

# Validation yap
results = model.val(
    data='/content/Obstacle-detection-7/data.yaml',
    imgsz=416,
    verbose=True
)

print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/gorme_engelli/obstacle_v2_yolov8m-4/weights/best.pt')